In [1]:
################
# Tabular ARGN #
################

## Authors:             Ricardo Zambrano
## email:               rzambrano@gmail.com
## Date:               2026-03-14

## reference:           "TabularARGN: A flexible and Efficient Auto-Regressive Framework for Generating high-Fidelity Synthetic Data"
##                      Tiwald et al. (2025)
##                      https://arxiv.org/pdf/2501.12012

######################
# Tabular ARGN       #
######################

## Author:             Ricardo Zambrano
## Email:              rzambrano@gmail.com
## Date:               2026-03-14
## Reference:          "TabularARGN: A Flexible and Efficient Auto-Regressive Framework
##                      for Generating High-Fidelity Synthetic Data"
##                      Tiwald et al. (2025)
##                      https://arxiv.org/abs/[arxiv-id]  # fill this in

#################
# Session Setup #
#################

# ── Standard Library ──────────────────────────────────────────────────────────
import os
import random
from pathlib import Path

from typing import Protocol

from datetime import date, datetime

# ── Numerical & Data ──────────────────────────────────────────────────────────
import numpy as np
import polars as pl
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Experiment Tracking ───────────────────────────────────────────────────────
import wandb
import mlflow
from omegaconf import DictConfig, OmegaConf
import hydra

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import f1_score, roc_auc_score
import optuna

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ── Hardcoded Variables ────────────────────────────────────────────────────────

# -- None --


c:\Users\rzamb\Documents\tabular_nn\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Working Directory ──────────────────────────────────────────────────────────

current_dir = Path.cwd().resolve()
print(current_dir)

target_dir = Path(r"c:\Users\rzamb\Documents\tabular_nn").resolve()

if current_dir != target_dir:

    # Get the path object for two levels up
    two_levels_up = current_dir.parents[0]

    # Change the working directory to the new path
    os.chdir(two_levels_up)

# Optional: Print new working directory to confirm the change
print(f"New working directory: {Path.cwd()}")

C:\Users\rzamb\Documents\tabular_nn\notebooks
New working directory: C:\Users\rzamb\Documents\tabular_nn


In [3]:
####################
# Helper Functions #
####################

# -- None --

In [4]:
####################
# Data Loading     #
####################

insurance_data = pd.read_csv("./data/raw/insurance/insurance.csv")

In [5]:
insurance_data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [6]:
insurance_data.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [7]:
insurance_data.dtypes

age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

In [8]:
beijing_data = pd.read_csv("./data/raw/beijing/PRSA_Data_Dingling_20130301-20170228.csv")

In [9]:
beijing_data.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,Dingling
1,2,2013,3,1,1,7.0,7.0,3.0,NaN,200.0,80.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Dingling
2,3,2013,3,1,2,5.0,5.0,3.0,2.0,200.0,79.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Dingling
3,4,2013,3,1,3,6.0,6.0,3.0,NaN,200.0,79.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Dingling
4,5,2013,3,1,4,5.0,5.0,3.0,NaN,200.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Dingling


In [10]:
beijing_data["sample_date"] = pd.to_datetime({
    'year': beijing_data['year'],
    'month': beijing_data['month'],
    'day': beijing_data['day']
})

In [11]:
beijing_data = beijing_data.drop(columns=["year", "month", "day"])

In [12]:
beijing_data.dtypes

No                      int64
hour                    int64
PM2.5                 float64
PM10                  float64
SO2                   float64
NO2                   float64
CO                    float64
O3                    float64
TEMP                  float64
PRES                  float64
DEWP                  float64
RAIN                  float64
wd                     object
WSPM                  float64
station                object
sample_date    datetime64[ns]
dtype: object

In [13]:
####################
# Data Wrangling   #
####################

In [14]:
# Introducing a null values in different formats

insurance_data.iloc[0,0] = pd.NA
insurance_data.iloc[1,0] = np.nan
insurance_data.iloc[2,1] = None
insurance_data.iloc[3,3] = 14

In [15]:
df_pl = pl.from_pandas(insurance_data).clone()

In [16]:
# Inserting an artificial person with a number of children disconnected   from the sequence 1->5
df_pl[3,3] = 14

In [17]:
# Inspecting how Polars casted missing values in a variety of formats

df_pl.head()

age,sex,bmi,children,smoker,region,charges
f64,str,f64,i64,str,str,f64
null,"""female""",27.9,0,"""yes""","""southwest""",16884.924
null,"""male""",33.77,1,"""no""","""southeast""",1725.5523
28.0,null,33.0,3,"""no""","""southeast""",4449.462
33.0,"""male""",22.705,14,"""no""","""northwest""",21984.47061
32.0,"""male""",28.88,0,"""no""","""northwest""",3866.8552


In [18]:
df_pl.null_count()

age,sex,bmi,children,smoker,region,charges
u32,u32,u32,u32,u32,u32,u32
2,1,0,0,0,0,0


In [19]:
nrow = df_pl.height

In [20]:
col_names = df_pl.columns

In [21]:
col_names

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

In [22]:
df_dtypes = [str(df_pl[name].dtype) for name in col_names]

In [23]:
df_dtypes

['Float64', 'String', 'Float64', 'Int64', 'String', 'String', 'Float64']

In [24]:
for i,dtype in enumerate(df_dtypes):
    if dtype in ["String", "Categorical", "Categories", "Enum", "Utf8"]:
        print(i,dtype)

1 String
4 String
5 String


In [25]:
df_pl[:,1].unique().to_list()

['female', None, 'male']

In [26]:
categorical_encode_maps = {}

for i,dtype in enumerate(df_dtypes):
    if dtype in ["String", "Categorical", "Categories", "Enum", "Utf8"]:
        print(i,col_names[i],df_pl[:,i].unique().to_list())

        unique_vals = df_pl[:, i].unique().to_list()
        col = col_names[i]

        if len(unique_vals) > nrow/3:
            Warning(f"Check {col} does not contain open ended values. This implementation only process categorical levels...")

        map_name = col + "_map"

        categorical_encode_maps[map_name] = {
            val: idx for idx, val in enumerate(unique_vals)
        }


1 sex ['male', 'female', None]
4 smoker ['no', 'yes']
5 region ['northeast', 'southwest', 'northwest', 'southeast']


In [27]:
categorical_encode_maps

{'sex_map': {'female': 0, None: 1, 'male': 2},
 'smoker_map': {'no': 0, 'yes': 1},
 'region_map': {'southeast': 0,
  'southwest': 1,
  'northeast': 2,
  'northwest': 3}}

In [28]:
len(categorical_encode_maps)

3

In [29]:
categorical_decode_maps = {
    outer_key: {v: k for k, v in inner_dict.items()}
    for outer_key, inner_dict in categorical_encode_maps.items()
}

In [30]:
categorical_decode_maps

{'sex_map': {0: 'female', 1: None, 2: 'male'},
 'smoker_map': {0: 'no', 1: 'yes'},
 'region_map': {0: 'southeast',
  1: 'southwest',
  2: 'northeast',
  3: 'northwest'}}

In [31]:
import importlib
import src.utils.tabular_datasets as tabular_datasets
importlib.reload(tabular_datasets)
from src.utils.tabular_datasets import argn_dataset

In [32]:
insurance_table_test = argn_dataset(insurance_data, clip_cols = True)

In [33]:
dir(insurance_table_test)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_bool_columns',
 '_categorical_columns',
 '_datetime_columns',
 '_float_columns_to_cast_to_integer',
 '_incompatible_columns',
 '_is_protocol',
 '_is_runtime_protocol',
 '_numeric_strategy',
 '_numerical_discrete_columns',
 '_numerical_float_columns',
 '_raw_data',
 '_table',
 '_table_pd',
 'argn_preprocessing',
 'categorical_decoding_maps',
 'categorical_encoding_maps',
 'clip_cols',
 'col_names',
 'dtypes',
 'load_data',
 'ncol',
 'nrow',
 'numerical_discrete_decoding_maps',
 'numerical_disc

In [34]:
insurance_data.head(10)

,age,sex,bmi,children,smoker,region,charges
0,NaN,female,27.900,0,yes,southwest,16884.92400
1,NaN,male,33.770,1,no,southeast,1725.55230
2,28.0,None,33.000,3,no,southeast,4449.46200
3,33.0,male,22.705,14,no,northwest,21984.47061
4,32.0,male,28.880,0,no,northwest,3866.85520
5,31.0,female,25.740,0,no,southeast,3756.62160
6,46.0,female,33.440,1,no,southeast,8240.58960
7,37.0,female,27.740,3,no,northwest,7281.50560
8,37.0,male,29.830,2,no,northeast,6406.41070
9,60.0,female,25.840,0,no,northwest,28923.13692


In [35]:
insurance_table_test.table.head(10)

,age,sex,bmi,children,smoker,region,charges
0,0,2,27.900,0,1,3,16884.92400
1,0,1,33.770,1,0,0,1725.55230
2,11,0,33.000,3,0,0,4449.46200
3,16,1,22.705,5,0,2,21984.47061
4,15,1,28.880,0,0,2,3866.85520
5,14,2,25.740,0,0,0,3756.62160
6,29,2,33.440,1,0,0,8240.58960
7,20,2,27.740,3,0,2,7281.50560
8,20,1,29.830,2,0,1,6406.41070
9,43,2,25.840,0,0,2,28923.13692


In [36]:
insurance_table_test._categorical_columns


[('sex', 1), ('smoker', 4), ('region', 5)]

In [37]:
c,i = zip(*insurance_table_test._categorical_columns)
print(c)
print(i)

('sex', 'smoker', 'region')
(1, 4, 5)


In [38]:
insurance_table_test._float_columns_to_cast_to_integer

[('age', 0)]

In [39]:
insurance_table_test._table

age,sex,bmi,children,smoker,region,charges
i64,i64,f64,i64,i64,i64,f64
0,2,27.9,0,1,3,16884.924
0,1,33.77,1,0,0,1725.5523
11,0,33.0,3,0,0,4449.462
16,1,22.705,5,0,2,21984.47061
15,1,28.88,0,0,2,3866.8552
…,…,…,…,…,…,…
33,1,30.97,3,0,2,10600.5483
1,2,31.92,0,0,1,2205.9808
1,2,36.85,0,0,0,1629.8335


In [40]:
insurance_table_test.numerical_discrete_encoding_maps

{'age': {None: 0,
  18: 1,
  19: 2,
  20: 3,
  21: 4,
  22: 5,
  23: 6,
  24: 7,
  25: 8,
  26: 9,
  27: 10,
  28: 11,
  29: 12,
  30: 13,
  31: 14,
  32: 15,
  33: 16,
  34: 17,
  35: 18,
  36: 19,
  37: 20,
  38: 21,
  39: 22,
  40: 23,
  41: 24,
  42: 25,
  43: 26,
  44: 27,
  45: 28,
  46: 29,
  47: 30,
  48: 31,
  49: 32,
  50: 33,
  51: 34,
  52: 35,
  53: 36,
  54: 37,
  55: 38,
  56: 39,
  57: 40,
  58: 41,
  59: 42,
  60: 43,
  61: 44,
  62: 45,
  63: 46,
  64: 47},
 'children': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5}}

In [41]:
insurance_table_test._incompatible_columns

[]

In [42]:
columns_with_float, _ = zip(*insurance_table_test._numerical_float_columns)

In [43]:
small_numbers = insurance_table_test._table['bmi'].to_numpy()
big_bumbers = insurance_table_test._table['charges'].to_numpy()

In [44]:
insurance_table_test._numeric_strategy

{'bmi': 'DIGIT', 'charges': 'DIGIT'}

In [45]:
insurance_table_test._numerical_float_columns

[('bmi', 2), ('charges', 6)]

In [46]:
lower = np.quantile(small_numbers, 0.01)
upper = np.quantile(small_numbers, 0.99)

print(f"lower = {lower} - upper = {upper}")

clipped = np.clip(small_numbers, lower, upper)

np.sort(small_numbers[np.logical_not(small_numbers == clipped)])

lower = 17.917294499999997 - upper = 46.330976999999955


array([17.89515, 17.89515, 17.89515, 17.89515, 17.89515, 17.89515,
       17.89515, 17.89515, 17.89515, 17.89515, 17.89515, 17.89515,
       17.89515, 17.89515, 46.4079 , 46.4079 , 46.4079 , 46.4079 ,
       46.4079 , 46.4079 , 46.4079 , 46.4079 , 46.4079 , 46.4079 ,
       46.4079 , 46.4079 , 46.4079 , 46.4079 ])

In [ ]:
def clip_columns(df_pl: pl.DataFrame, cols_to_process: list[tuple[str,int]], lower_quantile: float = 0.01, upper_quantile: float = 0.99) -> pl.DataFrame:
    """
    Clips columns with numeical values.

    Parameters:
    ----------

    df_pl: pl.DataFrame
        A polars data aframe
    cols_to_process: list[tuple[str,int]]
        Lis of columns with either integer or float values to beclipped
    lower_quantile: int, default_value = 0.01
        Lower quantile threshold. Values below this quantile are replaced 
        with the value at this quantile. 
        Setting the lower cutoff at 0.01 selects the value below
        which the lowest 1% of data points fall
    upper_quantile: int, default_value = 0.99
        Upper quantile threshold. Values above this quantile are replaced 
        with the value at this quantile.
        Setting the upper cutoff at 0.99 selects the value below
        which the lowest 99% of data points fall. It also means selecting
        a value above which the highest 1% of data points fall.

    Returns:
    -------

    processed_df : pl.DataFrame
        Polars data frame with specified columns clipped to the given quantile range
    """

    processed_df = df_pl
    cols_to_process_names, _ = zip(*cols_to_process)

    for col_name in cols_to_process_names:
        col_values = df_pl[col_name].to_numpy()
        lower = np.quantile(col_values, 0.01)
        upper = np.quantile(col_values, 0.99)
        processed_df.with_columns(pl.col("a").clip(lower, upper))

    return processed_df

    

In [64]:
test_clipping_df1 = pl.DataFrame({"a": [-50, 5, 50, None], "b": [150.92, 3, 12, 123]})
print(test_clipping_df1)

shape: (4, 2)
┌──────┬────────┐
│ a    ┆ b      │
│ ---  ┆ ---    │
│ i64  ┆ f64    │
╞══════╪════════╡
│ -50  ┆ 150.92 │
│ 5    ┆ 3.0    │
│ 50   ┆ 12.0   │
│ null ┆ 123.0  │
└──────┴────────┘


In [67]:
test_clipping_df2 =test_clipping_df1.with_columns(pl.col("a").clip(1, 10))
print(test_clipping_df2)

shape: (4, 2)
┌──────┬────────┐
│ a    ┆ b      │
│ ---  ┆ ---    │
│ i64  ┆ f64    │
╞══════╪════════╡
│ 1    ┆ 150.92 │
│ 5    ┆ 3.0    │
│ 10   ┆ 12.0   │
│ null ┆ 123.0  │
└──────┴────────┘


In [ ]:
def get_n_bins(df_pl: pl.DataFrame, float_cols: list[tuple[str,int]], max_bins: int = 100) -> dict[str,int]:
    """
    Calculates the cumber of bins
    """

In [50]:
def select_numeric_strategy(df_pl: pl.DataFrame, float_cols: list[tuple[str,int]]) -> list[str]:

    col_encoding_strategy = []

    for col in float_cols:

        numbers_to_analyze = df_pl[col].to_list()
        string_numbers = [str(abs(number)) for number in numbers_to_analyze] 
        splited_numbers = [tuple(str_number.split('.')) for str_number in string_numbers]
        len_int_and_dec = [(len(x), len(y)) for x, y in splited_numbers]
        
        max_decimal_places = max(x[1] for x in len_int_and_dec)

        max_num_digits = max((x[0]+x[1]) for x in len_int_and_dec)

        if max_decimal_places <=2:
            col_encoding_strategy.append("BINNED")
        elif max_num_digits > 6 or max_decimal_places > 3:
            col_encoding_strategy.append("DIGIT")
        else:
            col_encoding_strategy.append("BINNED")

    return col_encoding_strategy

In [49]:
columns_with_float

('bmi', 'charges')

In [51]:
select_numeric_strategy(insurance_table_test._table, columns_with_float)

['BINNED', 'DIGIT']

In [ ]:
# Numeric discrete

def generate_numeric_discrete_encoding_mappings():
    
    numerical_discrete_encoding_maps = {}
    numerical_discrete_columns = []

    for i,dtype in enumerate(df_dtypes):
        if dtype in ["Int8", "Int16", "Int32", "Int64", "UInt8", "UInt16", "UInt32", "UInt64"]:

            unique_vals = df_pl[:, i].unique().to_list()
            col = col_names[i]

In [ ]:
# Numeric Binned

In [ ]:
# Numeric Digit

In [ ]:
# Datetime